In [ ]:
# Packages for synthetic data generation and pre-processing.  
from sklearn.datasets import make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import pandas as pd

# Qiskit packages
from qiskit.circuit.library import ZZFeatureMap,RealAmplitudes
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.optimizers import SPSA
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Packages for data visualization
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

FILE_NAME = 'Heart_data.csv'
df = pd.read_csv(FILE_NAME)
print(f"SUCCESS: Data loaded from {FILE_NAME}")

FEATURE_COLUMNS = ['Age', 'Gender', 'BloodPressure', 'Cholesterol', 'HeartRate']
N_FEATURES = len(FEATURE_COLUMNS)

# X contains 5 features:
X = df[FEATURE_COLUMNS].values

# y contains the target label (HeartDisease):
y = df['HeartDisease'].values

# --- 3. Print Results ---
print("\n--- Data Head ---")
print(df.head()) # Shows the first 5 rows of the loaded DataFrame

print("\n--- Array Dimensions ---")
print(f"Features (X) shape: {X.shape}")
print(f"Labels (y) shape: {y.shape}")
print("\nData is now loaded from the external CSV and is ready for QML.")

print("\n--- Applying Min-Max Scaling to Features [0, 1] ---")
X_min = X.min(axis=0)
X_max = X.max(axis=0)
# Scale X such that all values are between 0 and 1
X_normalized = (X - X_min) / (X_max - X_min)
# Avoid division by zero if a column is constant
X_normalized[np.isnan(X_normalized)] = 0.0

TEST_SIZE = 0.2  # 30% of data for testing
RANDOM_SEED = 24 # Ensures reproducibility

X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, 
    y, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_SEED,
    # stratify=y  # Uncomment if you want proportional split of classes
)

print("\n--- Data Splitting Results ---")
print("Data successfully split into training and testing sets.")
print(f"Total Samples: {X.shape[0]}")
print(f"Training Samples (X_train): {X_train.shape[0]}")
print(f"Testing Samples (X_test): {X_test.shape[0]}")

train_data = X_train
test_data = X_test
train_labels = y_train
test_labels = y_test

feature_map = ZZFeatureMap(feature_dimension=N_FEATURES, reps=2)
feature_map.barrier()
var_form = RealAmplitudes(num_qubits=N_FEATURES, reps=4)
circuit = feature_map.compose(var_form).decompose()
circuit.measure_all()
circuit.decompose().draw(output="mpl", fold=-1, style="iqp")

def circuit_instance(data, variational):
    """
    Assigns parameter values to the quantum circuit.
    """
    parameters = {}
    for i, p in enumerate(feature_map.parameters):
        parameters[p] = data[i]
    for i, p in enumerate(var_form.ordered_parameters):
        parameters[p] = variational[i]
    return circuit.assign_parameters(parameters)

def labels_probability(results):
    """
    Converts a dictionary of bitstrings and their counts, to parities and their counts
    """
    shots = sum(results.values())
    probabilities = {0: 0, 1: 0}
    for bitstring, counts in results.items():
        label = (sum(int(k) for k in list(bitstring))+1) % 2
        probabilities[label] += counts / shots
    return probabilities

def classification_probability_simulation(data, variational):
    """
    Classify data points using given parameters.
    """
    circuits = [circuit_instance(d, variational) for d in data]
    sampler = StatevectorSampler() # Estimator() matrix_state_product
    results = sampler.run(circuits).result()
    # print(results[1].data.meas.get_counts())
    classification = [labels_probability(results[i].data.meas.get_counts()) for i, c in enumerate(circuits)]
    return classification

def cross_entropy_loss(classification, expected):
    """
    Calculate accuracy of predictions using cross entropy loss.
    """
    p = classification.get(expected)  
    return -np.log(p + 1e-10)

def cost_function(data, labels, variational):
    """
    Evaluates performance of our circuit with variational parameters on data.
    """
    classifications = classification_probability_simulation(data, variational)
    cost = 0
    for i, classification in enumerate(classifications):
        cost += cross_entropy_loss(classification, labels[i])
    cost /= len(data)
    return cost

def objective_function(variational):
    """Cost function of circuit parameters on training data.
    The optimizer will attempt to minimize this."""
    return cost_function(train_data, train_labels, variational)

class OptimizerLog:
    """Log to store optimizer's intermediate results"""
    def __init__(self):
        self.evaluations = []
        self.parameters = []
        self.costs = []
    def update(self, evaluation, parameter, cost, _stepsize, _accept):
        """Save intermediate results. Optimizer passes five values
        but we ignore the last two."""
        self.evaluations.append(evaluation)
        self.parameters.append(parameter)
        self.costs.append(cost)
             
# Set up the optimization
log = OptimizerLog()
optimizer = SPSA(maxiter=200, callback=log.update)

initial_point = np.zeros((var_form.num_parameters))

# Run the optimization
result = optimizer.minimize(objective_function, initial_point)

opt_var = result.x
opt_value = result.fun

fig = plt.figure()
plt.plot(log.evaluations, log.costs)
plt.title('VQC Cost Function Minimization')
plt.xlabel('Steps')
plt.ylabel('Cost')
plt.grid(True)
plt.rc('axes', linewidth=2)  # Set the axes linewidth globally
ax = plt.gca()  # Get current axes
ax.spines['top'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
plt.show()